## Version 1

It calculates three levels of output:

1. **Point-wise features** for each timestamp
   such as smoothed temperature, short-term changes, daily z-scores, and simple interactions with activity/light.

2. **Daily skin-temperature summary**
   such as mean, SD, range, night-vs-day difference, evening rise, daily correlation with activity/light, and 24 h cosinor metrics.

3. **Overall circadian metrics**
   such as **IS**, **IV**, and **RA** from skin temperature across the whole dataset.



A few important notes:

* I assumed `TEMPERATURE` is your main skin-temperature column.
* I used:

  * **night = 22:00–06:00**
  * **day = 10:00–18:00**
  * **evening rise = mean(20:00–22:00) − mean(16:00–18:00)**
* These windows are easy to change in `date_time_features()`.

The most useful outputs for you will probably be:

* `TEMP_NIGHT_MINUS_DAY`
* `TEMP_EVENING_RISE`
* `COSINOR_MESOR`
* `COSINOR_AMPLITUDE`
* `COSINOR_ACROPHASE_H`
* `TEMP_ACTIVITY_CORR`
* `TEMP_LIGHT_CORR`
* `TEMP_IS`, `TEMP_IV`, `TEMP_RA`

If you want, I can now make a **second version tailored more specifically to chronobiology**, for example adding:

* probable sleep-window temperature features,
* temperature phase angle relative to light exposure,
* weekly stability metrics,
* or integration with your existing activity/light circadian metrics.


In [19]:
from pathlib import Path
import numpy as np
import pandas as pd


# =========================
# CONFIG
# =========================
DATA_PATH = "/Users/arahjou/Documents/APP_CIRCADIAN_MEDICINE_v6_Ollama/data/data_set_1.csv"

DATETIME_COL = "DATE/TIME"
TEMP_COL = "TEMPERATURE"
ACTIVITY_COL = "PIMn"
LIGHT_COL = "MELANOPIC.EDI"

OUTPUT_DIR = "skin_temperature_outputs"


# =========================
# IO
# =========================
def load(path=DATA_PATH):
    df = pd.read_csv(path)
    return df


# =========================
# TIME FEATURES
# =========================
def date_time_features(df):
    df = df.copy()

    df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], errors="coerce")
    df = df.dropna(subset=[DATETIME_COL]).sort_values(DATETIME_COL).reset_index(drop=True)

    iso = df[DATETIME_COL].dt.isocalendar()

    df["DATE"] = df[DATETIME_COL].dt.date
    df["TIME"] = df[DATETIME_COL].dt.time
    df["WEEK"] = iso.week.astype("Int64")
    df["YEAR_ISO"] = iso.year.astype("Int64")
    df["DAY_OF_WEEK"] = df[DATETIME_COL].dt.day_name()
    df["HOUR"] = df[DATETIME_COL].dt.hour
    df["MINUTE"] = df[DATETIME_COL].dt.minute
    df["MINUTE_OF_DAY"] = df["HOUR"] * 60 + df["MINUTE"]

    # Adjustable masks
    df["IS_NIGHT"] = ((df["HOUR"] >= 22) | (df["HOUR"] < 6)).astype(int)
    df["IS_DAY"] = ((df["HOUR"] >= 10) & (df["HOUR"] < 18)).astype(int)
    df["IS_EVENING"] = ((df["HOUR"] >= 20) & (df["HOUR"] < 22)).astype(int)
    df["IS_LATE_AFTERNOON"] = ((df["HOUR"] >= 16) & (df["HOUR"] < 18)).astype(int)

    return df


# =========================
# BASIC PREPROCESSING
# =========================
def preprocess(df):
    df = df.copy()

    # Ensure numeric
    for col in [TEMP_COL, ACTIVITY_COL, LIGHT_COL]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Replace infinite values
    df = df.replace([np.inf, -np.inf], np.nan)

    # Temperature interpolation (light touch)
    df[TEMP_COL] = df[TEMP_COL].interpolate(limit_direction="both")

    # Activity and light should not be negative
    if ACTIVITY_COL in df.columns:
        df[ACTIVITY_COL] = df[ACTIVITY_COL].clip(lower=0)

    if LIGHT_COL in df.columns:
        df[LIGHT_COL] = df[LIGHT_COL].clip(lower=0)
        df["LOG_" + LIGHT_COL] = np.log1p(df[LIGHT_COL])

    return df


# =========================
# HELPERS
# =========================
def infer_sampling_minutes(df):
    diffs = (
        df[DATETIME_COL]
        .sort_values()
        .diff()
        .dropna()
        .dt.total_seconds()
        .div(60)
    )
    diffs = diffs[diffs > 0]

    if len(diffs) == 0:
        return 1.0

    return float(diffs.median())


def safe_corr(x, y):
    valid = x.notna() & y.notna()
    if valid.sum() < 3:
        return np.nan
    return x[valid].corr(y[valid])


def rolling_mean_by_time(df, col, window="30min"):
    s = df.set_index(DATETIME_COL)[col]
    out = s.rolling(window=window, min_periods=1).mean()
    return out.to_numpy()


def rolling_std_by_time(df, col, window="60min"):
    s = df.set_index(DATETIME_COL)[col]
    out = s.rolling(window=window, min_periods=2).std()
    return out.to_numpy()


def rolling_median_by_time(df, col, window="60min"):
    s = df.set_index(DATETIME_COL)[col]
    out = s.rolling(window=window, min_periods=1).median()
    return out.to_numpy()


def shift_periods_for_minutes(df, minutes):
    sampling_minutes = infer_sampling_minutes(df)
    periods = max(int(round(minutes / sampling_minutes)), 1)
    return periods


# =========================
# POINT-WISE SKIN TEMP FEATURES
# =========================
def add_skin_temp_features(df):
    df = df.copy()

    # Smoothed temperature
    df["TEMP_SMOOTH_30MIN"] = rolling_mean_by_time(df, TEMP_COL, "30min")
    df["TEMP_SMOOTH_60MIN"] = rolling_mean_by_time(df, TEMP_COL, "60min")
    df["TEMP_SD_60MIN"] = rolling_std_by_time(df, TEMP_COL, "60min")
    df["TEMP_MEDIAN_60MIN"] = rolling_median_by_time(df, TEMP_COL, "60min")

    # Activity/light smoothing for contextual coupling
    if ACTIVITY_COL in df.columns:
        df["ACTIVITY_SMOOTH_30MIN"] = rolling_mean_by_time(df, ACTIVITY_COL, "30min")
        df["ACTIVITY_SMOOTH_60MIN"] = rolling_mean_by_time(df, ACTIVITY_COL, "60min")

    if LIGHT_COL in df.columns:
        df["LIGHT_SMOOTH_30MIN"] = rolling_mean_by_time(df, LIGHT_COL, "30min")
        df["LIGHT_SMOOTH_60MIN"] = rolling_mean_by_time(df, LIGHT_COL, "60min")
        df["LOG_LIGHT_SMOOTH_30MIN"] = np.log1p(df["LIGHT_SMOOTH_30MIN"])
        df["LOG_LIGHT_SMOOTH_60MIN"] = np.log1p(df["LIGHT_SMOOTH_60MIN"])

    # Short-term temperature change
    for minutes in [10, 30, 60]:
        p = shift_periods_for_minutes(df, minutes)
        df[f"TEMP_DELTA_{minutes}MIN"] = df[TEMP_COL] - df[TEMP_COL].shift(p)

    # Day-normalized temperature
    df["TEMP_DAILY_MEAN"] = df.groupby("DATE")[TEMP_COL].transform("mean")
    df["TEMP_DAILY_SD"] = df.groupby("DATE")[TEMP_COL].transform("std")
    df["TEMP_CENTERED_DAY"] = df[TEMP_COL] - df["TEMP_DAILY_MEAN"]
    df["TEMP_Z_DAY"] = np.where(
        df["TEMP_DAILY_SD"] > 0,
        (df[TEMP_COL] - df["TEMP_DAILY_MEAN"]) / df["TEMP_DAILY_SD"],
        np.nan
    )

    # Simple context interactions
    if ACTIVITY_COL in df.columns:
        df["TEMP_X_ACTIVITY"] = df["TEMP_SMOOTH_30MIN"] * df["ACTIVITY_SMOOTH_30MIN"]

    if LIGHT_COL in df.columns:
        df["TEMP_X_LOG_LIGHT"] = df["TEMP_SMOOTH_30MIN"] * df["LOG_LIGHT_SMOOTH_30MIN"]

    return df


# =========================
# 24H COSINOR FOR A SINGLE DAY
# =========================
def fit_24h_cosinor(day_df, value_col=TEMP_COL):
    """
    Fits: y = M + b*cos(wt) + c*sin(wt), with period fixed to 24 h
    Returns mesor, amplitude, acrophase_hours, r2
    """
    tmp = day_df[[value_col, "MINUTE_OF_DAY"]].dropna().copy()

    if len(tmp) < 12 or tmp["MINUTE_OF_DAY"].nunique() < 6:
        return {
            "COSINOR_MESOR": np.nan,
            "COSINOR_AMPLITUDE": np.nan,
            "COSINOR_ACROPHASE_H": np.nan,
            "COSINOR_R2": np.nan,
        }

    t_hours = tmp["MINUTE_OF_DAY"].to_numpy() / 60.0
    y = tmp[value_col].to_numpy()

    w = 2 * np.pi / 24.0
    X = np.column_stack([
        np.ones(len(t_hours)),
        np.cos(w * t_hours),
        np.sin(w * t_hours)
    ])

    try:
        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        mesor = beta[0]
        b = beta[1]
        c = beta[2]
        amplitude = np.sqrt(b**2 + c**2)

        # Peak time
        phi = np.arctan2(c, b)
        if phi < 0:
            phi += 2 * np.pi
        acrophase_h = 24.0 * phi / (2 * np.pi)

        y_hat = X @ beta
        ss_res = np.sum((y - y_hat) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        return {
            "COSINOR_MESOR": mesor,
            "COSINOR_AMPLITUDE": amplitude,
            "COSINOR_ACROPHASE_H": acrophase_h,
            "COSINOR_R2": r2,
        }

    except Exception:
        return {
            "COSINOR_MESOR": np.nan,
            "COSINOR_AMPLITUDE": np.nan,
            "COSINOR_ACROPHASE_H": np.nan,
            "COSINOR_R2": np.nan,
        }


# =========================
# DAILY SUMMARY
# =========================
def summarize_one_day(day_df):
    out = {}

    temp = day_df[TEMP_COL].dropna()

    if len(temp) == 0:
        return pd.Series(dtype="float64")

    out["N_OBS"] = len(temp)
    out["TEMP_MEAN"] = temp.mean()
    out["TEMP_MEDIAN"] = temp.median()
    out["TEMP_SD"] = temp.std()
    out["TEMP_MIN"] = temp.min()
    out["TEMP_MAX"] = temp.max()
    out["TEMP_RANGE"] = temp.max() - temp.min()
    out["TEMP_P05"] = temp.quantile(0.05)
    out["TEMP_P95"] = temp.quantile(0.95)
    out["TEMP_ROBUST_AMPLITUDE"] = (out["TEMP_P95"] - out["TEMP_P05"]) / 2

    # Day vs night
    temp_day = day_df.loc[day_df["IS_DAY"] == 1, TEMP_COL].dropna()
    temp_night = day_df.loc[day_df["IS_NIGHT"] == 1, TEMP_COL].dropna()

    out["TEMP_DAY_MEAN"] = temp_day.mean() if len(temp_day) else np.nan
    out["TEMP_NIGHT_MEAN"] = temp_night.mean() if len(temp_night) else np.nan
    out["TEMP_NIGHT_MINUS_DAY"] = (
        out["TEMP_NIGHT_MEAN"] - out["TEMP_DAY_MEAN"]
        if pd.notna(out["TEMP_NIGHT_MEAN"]) and pd.notna(out["TEMP_DAY_MEAN"])
        else np.nan
    )

    # Evening rise: 20-22 minus 16-18
    temp_evening = day_df.loc[day_df["IS_EVENING"] == 1, TEMP_COL].dropna()
    temp_late_afternoon = day_df.loc[day_df["IS_LATE_AFTERNOON"] == 1, TEMP_COL].dropna()

    out["TEMP_EVENING_MEAN"] = temp_evening.mean() if len(temp_evening) else np.nan
    out["TEMP_LATE_AFTERNOON_MEAN"] = temp_late_afternoon.mean() if len(temp_late_afternoon) else np.nan
    out["TEMP_EVENING_RISE"] = (
        out["TEMP_EVENING_MEAN"] - out["TEMP_LATE_AFTERNOON_MEAN"]
        if pd.notna(out["TEMP_EVENING_MEAN"]) and pd.notna(out["TEMP_LATE_AFTERNOON_MEAN"])
        else np.nan
    )

    # Coupling with activity and light
    if ACTIVITY_COL in day_df.columns:
        out["TEMP_ACTIVITY_CORR"] = safe_corr(day_df[TEMP_COL], day_df[ACTIVITY_COL])
        out["ACTIVITY_SUM"] = day_df[ACTIVITY_COL].sum()
        out["ACTIVITY_MEAN"] = day_df[ACTIVITY_COL].mean()

    if LIGHT_COL in day_df.columns:
        out["TEMP_LIGHT_CORR"] = safe_corr(day_df[TEMP_COL], day_df[LIGHT_COL])
        out["TEMP_LOG_LIGHT_CORR"] = safe_corr(day_df[TEMP_COL], np.log1p(day_df[LIGHT_COL]))
        out["LIGHT_SUM"] = day_df[LIGHT_COL].sum()
        out["LIGHT_MEAN"] = day_df[LIGHT_COL].mean()

        light_day = day_df.loc[day_df["IS_DAY"] == 1, LIGHT_COL]
        light_night = day_df.loc[day_df["IS_NIGHT"] == 1, LIGHT_COL]
        out["LIGHT_DAY_SUM"] = light_day.sum()
        out["LIGHT_NIGHT_SUM"] = light_night.sum()

    # 24h cosinor
    out.update(fit_24h_cosinor(day_df, TEMP_COL))

    return pd.Series(out)


def compute_daily_skin_temp_summary(df):
    daily = (
        df.groupby("DATE", group_keys=False)
        .apply(summarize_one_day)
        .reset_index()
    )

    if len(daily) == 0:
        return daily

    daily["DATE"] = pd.to_datetime(daily["DATE"])
    daily = daily.sort_values("DATE").reset_index(drop=True)

    # Rolling 7-day baseline for anomaly / illness-like deviation
    for col in ["TEMP_MEAN", "TEMP_ROBUST_AMPLITUDE", "TEMP_NIGHT_MINUS_DAY"]:
        if col in daily.columns:
            baseline = daily[col].rolling(7, min_periods=3).mean()
            baseline_sd = daily[col].rolling(7, min_periods=3).std()

            daily[f"{col}_BASELINE_7D"] = baseline
            daily[f"{col}_DELTA_FROM_7D"] = daily[col] - baseline
            daily[f"{col}_Z_7D"] = np.where(
                baseline_sd > 0,
                (daily[col] - baseline) / baseline_sd,
                np.nan
            )

    return daily


# =========================
# OVERALL CIRCADIAN METRICS
# =========================
def compute_overall_temp_circadian_metrics(df):
    """
    Computes non-parametric circadian rhythm metrics from hourly skin temperature:
    - IS: interdaily stability
    - IV: intradaily variability
    - RA: relative amplitude using highest 5h and lowest 10h means
    """
    s = (
        df.set_index(DATETIME_COL)[TEMP_COL]
        .resample("1h")
        .mean()
        .dropna()
    )

    if len(s) < 48:
        return {
            "TEMP_IS": np.nan,
            "TEMP_IV": np.nan,
            "TEMP_RA": np.nan,
            "TEMP_H5": np.nan,
            "TEMP_L10": np.nan,
            "N_HOURLY_POINTS": len(s),
        }

    x = s.to_numpy()
    N = len(x)
    grand_mean = np.mean(x)

    denom = np.sum((x - grand_mean) ** 2)

    # Interdaily Stability (IS)
    mean_by_clock_hour = s.groupby(s.index.hour).mean()
    p = 24
    if denom > 0:
        IS = (N * np.sum((mean_by_clock_hour - grand_mean) ** 2)) / (p * denom)
    else:
        IS = np.nan

    # Intradaily Variability (IV)
    if denom > 0 and N > 1:
        IV = (N * np.sum(np.diff(x) ** 2)) / ((N - 1) * denom)
    else:
        IV = np.nan

    # Relative Amplitude (RA)
    H5 = s.rolling(5, min_periods=5).mean().max()
    L10 = s.rolling(10, min_periods=10).mean().min()

    if pd.notna(H5) and pd.notna(L10) and (H5 + L10) != 0:
        RA = (H5 - L10) / (H5 + L10)
    else:
        RA = np.nan

    return {
        "TEMP_IS": IS,
        "TEMP_IV": IV,
        "TEMP_RA": RA,
        "TEMP_H5": H5,
        "TEMP_L10": L10,
        "N_HOURLY_POINTS": N,
    }


# =========================
# MAIN
# =========================
def main():
    # Load and prepare
    df = load()
    df = date_time_features(df)
    df = preprocess(df)

    # Add point-wise features
    df_features = add_skin_temp_features(df)

    # Daily summary
    daily_summary = compute_daily_skin_temp_summary(df_features)

    # Overall summary
    overall_summary = compute_overall_temp_circadian_metrics(df_features)
    overall_summary_df = pd.DataFrame([overall_summary])

    # Output
    outdir = Path(OUTPUT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)

    #df_features.to_csv(outdir / "data_with_skin_temperature_features.csv", index=False)
    #daily_summary.to_csv(outdir / "daily_skin_temperature_summary.csv", index=False)
    #overall_summary_df.to_csv(outdir / "overall_skin_temperature_circadian_metrics.csv", index=False)

    # Show results
    print("\nColumns in dataframe:")
    print(df_features.columns.tolist())

    print("\nHead of point-wise feature dataframe:")
    print(df_features.head())

    print("\nHead of daily summary:")
    print(daily_summary.head())

    print("\nOverall circadian metrics from skin temperature:")
    print(overall_summary_df)


if __name__ == "__main__":
    main()


Columns in dataframe:
['DATE/TIME', 'MS', 'EVENT', 'TEMPERATURE', 'EXT.TEMPERATURE', 'ORIENTATION', 'PIM', 'PIMn', 'TAT', 'TATn', 'ZCM', 'ZCMn', 'LIGHT', 'AMB.LIGHT', 'RED.LIGHT', 'GREEN.LIGHT', 'BLUE.LIGHT', 'IR.LIGHT', 'UVA.LIGHT', 'UVB.LIGHT', 'STATE', 'CAP_SENS_1', 'CAP_SENS_2', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'MELANOPIC.EDI', 'CLEAR', 'SLEEP_STATE', 'DATE', 'TIME', 'WEEK', 'YEAR_ISO', 'DAY_OF_WEEK', 'HOUR', 'MINUTE', 'MINUTE_OF_DAY', 'IS_NIGHT', 'IS_DAY', 'IS_EVENING', 'IS_LATE_AFTERNOON', 'LOG_MELANOPIC.EDI', 'TEMP_SMOOTH_30MIN', 'TEMP_SMOOTH_60MIN', 'TEMP_SD_60MIN', 'TEMP_MEDIAN_60MIN', 'ACTIVITY_SMOOTH_30MIN', 'ACTIVITY_SMOOTH_60MIN', 'LIGHT_SMOOTH_30MIN', 'LIGHT_SMOOTH_60MIN', 'LOG_LIGHT_SMOOTH_30MIN', 'LOG_LIGHT_SMOOTH_60MIN', 'TEMP_DELTA_10MIN', 'TEMP_DELTA_30MIN', 'TEMP_DELTA_60MIN', 'TEMP_DAILY_MEAN', 'TEMP_DAILY_SD', 'TEMP_CENTERED_DAY', 'TEMP_Z_DAY', 'TEMP_X_ACTIVITY', 'TEMP_X_LOG_LIGHT']

Head of point-wise feature dataframe:
            DATE/TIME  MS  

/var/folders/j5/c5vt47v9695609y1ssl7zjdr0000gn/T/ipykernel_39897/2561134788.py:317: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_one_day)


## A second version tailored more specifically to chronobiology

This version focuses less on generic signal description and more on **circadian interpretation** of skin temperature in relation to:

* **skin temperature rhythm**
* **activity-rest behavior**
* **melanopic light exposure**
* **weekday vs weekend / social timing**
* **phase relationships between signals**

It is designed for the kind of longitudinal wearable data you are working with.

### What this version adds

Compared with the previous script, this one emphasizes chronobiology-oriented metrics such as:

* **daily temperature mesor, amplitude, acrophase**
* **daily temperature maximum and minimum timing**
* **night-day temperature contrast**
* **evening temperature rise**
* **warming onset time**
* **phase angle between temperature and light**
* **phase angle between temperature and activity**
* **probable rest window proxy** from low activity + low light + high temperature
* **weekday vs weekend phase differences**
* **overall rhythm stability and fragmentation**
* **relative amplitude based on hottest/coldest windows**
* **daily “circadian alignment” indicators**

A key point: with only one peripheral skin temperature signal, these are still **proxies**, not direct measurements of core circadian phase. But they are much more chronobiology-oriented than simple mean or SD.

---

## Why this version is more chronobiology-specific

### 1. It treats skin temperature as a **circadian signal**

Instead of only using mean and SD, it extracts:

* `TEMP_COS_ACROPHASE_H`
* `TEMP_COS_AMPLITUDE`
* `TEMP_WARMING_ONSET_H`
* `TEMP_MAX_H`
* `TEMP_MIN_H`

These are much closer to circadian interpretation.

---

### 2. It examines **phase relationships**

This is often more informative than temperature alone.

Examples:

* `PHASE_TEMP_TO_LIGHT_PEAK_H`
* `PHASE_TEMP_TO_ACTIVITY_PEAK_H`
* `PHASE_WARMING_TO_LIGHT_OFFSET_H`
* `PHASE_WARMING_TO_ACTIVITY_OFFSET_H`
* `PHASE_TEMP_TO_REST_MIDPOINT_H`

These help you study whether the temperature rhythm is aligned or shifted relative to behavior and light exposure.

---

### 3. It includes a **rest-phase proxy**

Because you do not have PSG in this dataframe, the script uses:

* low activity
* low light
* high temperature

to estimate a probable rest window:

* `REST_ONSET_H_PROXY`
* `REST_OFFSET_H_PROXY`
* `REST_MIDPOINT_H_PROXY`
* `REST_DURATION_H_PROXY`

That is not sleep scoring, but it is often useful in chronobiology as a behavioral anchor.

---

### 4. It includes **weekday vs weekend misalignment**

This is useful for social jetlag-like effects.

The script outputs weekday/weekend differences for:

* temperature acrophase
* warming onset
* rest midpoint
* activity offset
* light offset

---

## Metrics I would pay most attention to first

For your type of work, these are probably the strongest candidates:

* `TEMP_COS_ACROPHASE_H`
* `TEMP_COS_AMPLITUDE`
* `TEMP_NIGHT_MINUS_DAY`
* `TEMP_EVENING_RISE`
* `TEMP_WARMING_ONSET_H`
* `REST_MIDPOINT_H_PROXY`
* `PHASE_WARMING_TO_LIGHT_OFFSET_H`
* `PHASE_WARMING_TO_ACTIVITY_OFFSET_H`
* `TEMP_IS`
* `TEMP_IV`

---

## A few scientific cautions

This version is much more useful biologically, but still keep in mind:

* peripheral skin temperature is strongly affected by **ambient conditions**
* it is also affected by **movement**, **clothing**, **contact**, and **sensor placement**
* the rest proxy is not equivalent to PSG sleep
* light thresholds and activity thresholds are currently **data-driven**, not yet physiologically optimized

So this is best treated as a **feature extraction framework**, not a final biological truth layer.

---

## Good next step

The next refinement I would recommend is a **third version** where these temperature metrics are explicitly integrated with your existing circadian metrics for activity and light, so you get derived variables such as:

* temperature-light phase angle
* temperature-activity phase angle
* alignment score per day
* circadian disruption score
* probable internal misalignment index

That would fit very well with your broader chronomedicine pipeline.


In [20]:
from pathlib import Path
import numpy as np
import pandas as pd


# ==========================================
# CONFIG
# ==========================================
DATA_PATH = "/Users/arahjou/Documents/APP_CIRCADIAN_MEDICINE_v6_Ollama/data/data_set_1.csv"

DATETIME_COL = "DATE/TIME"
TEMP_COL = "TEMPERATURE"
ACTIVITY_COL = "PIMn"
LIGHT_COL = "MELANOPIC.EDI"

OUTPUT_DIR = "skin_temp_chronobio_outputs"

# You can tune these thresholds later if needed
EVENING_START = 18
EVENING_END = 24
DAY_START = 10
DAY_END = 18
NIGHT_START = 22
NIGHT_END = 6

REST_ACTIVITY_QUANTILE = 0.30
REST_LIGHT_QUANTILE = 0.30
REST_TEMP_QUANTILE = 0.70


# ==========================================
# IO
# ==========================================
def load(path=DATA_PATH):
    df = pd.read_csv(path)
    return df


# ==========================================
# DATETIME FEATURES
# ==========================================
def add_datetime_features(df):
    df = df.copy()

    df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], errors="coerce")
    df = df.dropna(subset=[DATETIME_COL]).sort_values(DATETIME_COL).reset_index(drop=True)

    iso = df[DATETIME_COL].dt.isocalendar()

    df["DATE"] = df[DATETIME_COL].dt.date
    df["TIME"] = df[DATETIME_COL].dt.time
    df["WEEK"] = iso.week.astype("Int64")
    df["ISO_YEAR"] = iso.year.astype("Int64")
    df["DAY_OF_WEEK"] = df[DATETIME_COL].dt.day_name()
    df["WEEKDAY_NUM"] = df[DATETIME_COL].dt.weekday
    df["IS_WEEKEND"] = (df["WEEKDAY_NUM"] >= 5).astype(int)

    df["HOUR"] = df[DATETIME_COL].dt.hour
    df["MINUTE"] = df[DATETIME_COL].dt.minute
    df["SECOND"] = df[DATETIME_COL].dt.second
    df["MINUTE_OF_DAY"] = df["HOUR"] * 60 + df["MINUTE"] + df["SECOND"] / 60

    # Circadian time windows
    df["IS_DAY"] = ((df["HOUR"] >= DAY_START) & (df["HOUR"] < DAY_END)).astype(int)
    df["IS_EVENING"] = ((df["HOUR"] >= EVENING_START) & (df["HOUR"] < EVENING_END)).astype(int)
    df["IS_NIGHT"] = ((df["HOUR"] >= NIGHT_START) | (df["HOUR"] < NIGHT_END)).astype(int)

    return df


# ==========================================
# PREPROCESS
# ==========================================
def preprocess(df):
    df = df.copy()

    for col in [TEMP_COL, ACTIVITY_COL, LIGHT_COL]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.replace([np.inf, -np.inf], np.nan)

    # Light interpolation for temperature only
    df[TEMP_COL] = df[TEMP_COL].interpolate(limit_direction="both")

    if ACTIVITY_COL in df.columns:
        df[ACTIVITY_COL] = df[ACTIVITY_COL].clip(lower=0)

    if LIGHT_COL in df.columns:
        df[LIGHT_COL] = df[LIGHT_COL].clip(lower=0)
        df["LOG_LIGHT"] = np.log1p(df[LIGHT_COL])

    return df


# ==========================================
# HELPERS
# ==========================================
def infer_sampling_minutes(df):
    diffs = (
        df[DATETIME_COL]
        .sort_values()
        .diff()
        .dropna()
        .dt.total_seconds()
        .div(60)
    )
    diffs = diffs[diffs > 0]
    if len(diffs) == 0:
        return 1.0
    return float(diffs.median())


def safe_corr(x, y):
    valid = x.notna() & y.notna()
    if valid.sum() < 3:
        return np.nan
    return x[valid].corr(y[valid])


def circular_mean_hours(hours):
    hours = pd.Series(hours).dropna()
    if len(hours) == 0:
        return np.nan
    angles = 2 * np.pi * hours / 24
    sin_mean = np.mean(np.sin(angles))
    cos_mean = np.mean(np.cos(angles))
    angle = np.arctan2(sin_mean, cos_mean)
    if angle < 0:
        angle += 2 * np.pi
    return 24 * angle / (2 * np.pi)


def circular_diff_hours(h1, h2):
    """
    signed difference h1 - h2 in range [-12, 12]
    """
    if pd.isna(h1) or pd.isna(h2):
        return np.nan
    diff = ((h1 - h2 + 12) % 24) - 12
    return diff


def rolling_mean_time(df, col, window="30min"):
    s = df.set_index(DATETIME_COL)[col]
    return s.rolling(window=window, min_periods=1).mean().to_numpy()


def rolling_std_time(df, col, window="60min"):
    s = df.set_index(DATETIME_COL)[col]
    return s.rolling(window=window, min_periods=2).std().to_numpy()


def get_first_time_above_threshold(day_df, value_col, threshold, hour_start=18, hour_end=24):
    sub = day_df[(day_df["HOUR"] >= hour_start) & (day_df["HOUR"] < hour_end)].copy()
    sub = sub[[DATETIME_COL, "MINUTE_OF_DAY", value_col]].dropna()
    if len(sub) == 0:
        return np.nan
    hit = sub[sub[value_col] >= threshold]
    if len(hit) == 0:
        return np.nan
    return float(hit["MINUTE_OF_DAY"].iloc[0] / 60.0)


def get_daily_extreme_time(day_df, value_col, mode="max"):
    tmp = day_df[[value_col, "MINUTE_OF_DAY"]].dropna()
    if len(tmp) == 0:
        return np.nan, np.nan

    if mode == "max":
        idx = tmp[value_col].idxmax()
    else:
        idx = tmp[value_col].idxmin()

    value = day_df.loc[idx, value_col]
    time_h = day_df.loc[idx, "MINUTE_OF_DAY"] / 60.0
    return value, float(time_h)


def get_window_mean_extreme_time(day_df, value_col, window_hours=5, mode="max"):
    """
    Finds hottest/coldest rolling window timing.
    """
    tmp = day_df[[DATETIME_COL, value_col]].dropna().copy()
    if len(tmp) < 3:
        return np.nan, np.nan

    tmp = tmp.set_index(DATETIME_COL).sort_index()
    rolled = tmp[value_col].rolling(f"{int(window_hours)}h", min_periods=2).mean()

    if rolled.dropna().empty:
        return np.nan, np.nan

    if mode == "max":
        t = rolled.idxmax()
        val = rolled.max()
    else:
        t = rolled.idxmin()
        val = rolled.min()

    time_h = t.hour + t.minute / 60 + t.second / 3600
    return float(val), float(time_h)


# ==========================================
# POINTWISE FEATURES
# ==========================================
def add_pointwise_chronobio_features(df):
    df = df.copy()

    df["TEMP_SMOOTH_30MIN"] = rolling_mean_time(df, TEMP_COL, "30min")
    df["TEMP_SMOOTH_60MIN"] = rolling_mean_time(df, TEMP_COL, "60min")
    df["TEMP_SD_60MIN"] = rolling_std_time(df, TEMP_COL, "60min")

    if ACTIVITY_COL in df.columns:
        df["ACT_SMOOTH_30MIN"] = rolling_mean_time(df, ACTIVITY_COL, "30min")
        df["ACT_SMOOTH_60MIN"] = rolling_mean_time(df, ACTIVITY_COL, "60min")

    if LIGHT_COL in df.columns:
        df["LIGHT_SMOOTH_30MIN"] = rolling_mean_time(df, LIGHT_COL, "30min")
        df["LIGHT_SMOOTH_60MIN"] = rolling_mean_time(df, LIGHT_COL, "60min")
        df["LOG_LIGHT_SMOOTH_30MIN"] = np.log1p(df["LIGHT_SMOOTH_30MIN"])

    # Day-centered and z-scored temperature
    df["TEMP_DAY_MEAN_POINTWISE"] = df.groupby("DATE")[TEMP_COL].transform("mean")
    df["TEMP_DAY_SD_POINTWISE"] = df.groupby("DATE")[TEMP_COL].transform("std")
    df["TEMP_CENTERED_DAY"] = df[TEMP_COL] - df["TEMP_DAY_MEAN_POINTWISE"]
    df["TEMP_Z_DAY"] = np.where(
        df["TEMP_DAY_SD_POINTWISE"] > 0,
        (df[TEMP_COL] - df["TEMP_DAY_MEAN_POINTWISE"]) / df["TEMP_DAY_SD_POINTWISE"],
        np.nan
    )

    # Phase-related proxy: temperature high while activity/light low
    if ACTIVITY_COL in df.columns and LIGHT_COL in df.columns:
        df["REST_PROPENSITY_PROXY"] = (
            df["TEMP_Z_DAY"].fillna(0)
            - pd.Series(df["ACT_SMOOTH_30MIN"]).rank(pct=True).fillna(0).to_numpy()
            - pd.Series(df["LIGHT_SMOOTH_30MIN"]).rank(pct=True).fillna(0).to_numpy()
        )

    return df


# ==========================================
# COSINOR
# ==========================================
def fit_24h_cosinor(day_df, value_col=TEMP_COL):
    tmp = day_df[[value_col, "MINUTE_OF_DAY"]].dropna().copy()

    if len(tmp) < 12 or tmp["MINUTE_OF_DAY"].nunique() < 6:
        return {
            "TEMP_COS_MESOR": np.nan,
            "TEMP_COS_AMPLITUDE": np.nan,
            "TEMP_COS_ACROPHASE_H": np.nan,
            "TEMP_COS_R2": np.nan,
        }

    t_hours = tmp["MINUTE_OF_DAY"].to_numpy() / 60.0
    y = tmp[value_col].to_numpy()

    w = 2 * np.pi / 24.0
    X = np.column_stack([
        np.ones(len(t_hours)),
        np.cos(w * t_hours),
        np.sin(w * t_hours)
    ])

    try:
        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        mesor = beta[0]
        b = beta[1]
        c = beta[2]
        amplitude = np.sqrt(b ** 2 + c ** 2)

        phi = np.arctan2(c, b)
        if phi < 0:
            phi += 2 * np.pi
        acrophase_h = 24 * phi / (2 * np.pi)

        y_hat = X @ beta
        ss_res = np.sum((y - y_hat) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        return {
            "TEMP_COS_MESOR": mesor,
            "TEMP_COS_AMPLITUDE": amplitude,
            "TEMP_COS_ACROPHASE_H": acrophase_h,
            "TEMP_COS_R2": r2,
        }
    except Exception:
        return {
            "TEMP_COS_MESOR": np.nan,
            "TEMP_COS_AMPLITUDE": np.nan,
            "TEMP_COS_ACROPHASE_H": np.nan,
            "TEMP_COS_R2": np.nan,
        }


# ==========================================
# DAILY LIGHT / ACTIVITY PHASE MARKERS
# ==========================================
def compute_daily_light_markers(day_df):
    out = {
        "LIGHT_ONSET_H": np.nan,
        "LIGHT_OFFSET_H": np.nan,
        "LIGHT_MIDPOINT_H": np.nan,
        "LIGHT_PEAK_H": np.nan,
        "LIGHT_DAY_SUM": np.nan,
        "LIGHT_EVENING_SUM": np.nan,
        "EVENING_LIGHT_SUPPRESSION_INDEX": np.nan,
    }

    if LIGHT_COL not in day_df.columns:
        return out

    tmp = day_df[[DATETIME_COL, "MINUTE_OF_DAY", LIGHT_COL, "IS_DAY", "IS_EVENING"]].dropna().copy()
    if len(tmp) == 0:
        return out

    out["LIGHT_DAY_SUM"] = tmp.loc[tmp["IS_DAY"] == 1, LIGHT_COL].sum()
    out["LIGHT_EVENING_SUM"] = tmp.loc[tmp["IS_EVENING"] == 1, LIGHT_COL].sum()

    daily_thresh = tmp[LIGHT_COL].quantile(0.25)
    above = tmp[tmp[LIGHT_COL] > daily_thresh]

    if len(above) > 0:
        out["LIGHT_ONSET_H"] = float(above["MINUTE_OF_DAY"].iloc[0] / 60.0)
        out["LIGHT_OFFSET_H"] = float(above["MINUTE_OF_DAY"].iloc[-1] / 60.0)
        out["LIGHT_MIDPOINT_H"] = (out["LIGHT_ONSET_H"] + out["LIGHT_OFFSET_H"]) / 2.0

    peak_idx = tmp[LIGHT_COL].idxmax()
    out["LIGHT_PEAK_H"] = float(day_df.loc[peak_idx, "MINUTE_OF_DAY"] / 60.0)

    day_sum = out["LIGHT_DAY_SUM"]
    eve_sum = out["LIGHT_EVENING_SUM"]
    out["EVENING_LIGHT_SUPPRESSION_INDEX"] = eve_sum / day_sum if day_sum and day_sum > 0 else np.nan

    return out


def compute_daily_activity_markers(day_df):
    out = {
        "ACTIVITY_ONSET_H": np.nan,
        "ACTIVITY_OFFSET_H": np.nan,
        "ACTIVITY_MIDPOINT_H": np.nan,
        "ACTIVITY_PEAK_H": np.nan,
        "ACTIVITY_SUM": np.nan,
    }

    if ACTIVITY_COL not in day_df.columns:
        return out

    tmp = day_df[[DATETIME_COL, "MINUTE_OF_DAY", ACTIVITY_COL]].dropna().copy()
    if len(tmp) == 0:
        return out

    out["ACTIVITY_SUM"] = tmp[ACTIVITY_COL].sum()

    thresh = tmp[ACTIVITY_COL].quantile(0.30)
    active = tmp[tmp[ACTIVITY_COL] > thresh]

    if len(active) > 0:
        out["ACTIVITY_ONSET_H"] = float(active["MINUTE_OF_DAY"].iloc[0] / 60.0)
        out["ACTIVITY_OFFSET_H"] = float(active["MINUTE_OF_DAY"].iloc[-1] / 60.0)
        out["ACTIVITY_MIDPOINT_H"] = (out["ACTIVITY_ONSET_H"] + out["ACTIVITY_OFFSET_H"]) / 2.0

    peak_idx = tmp[ACTIVITY_COL].idxmax()
    out["ACTIVITY_PEAK_H"] = float(day_df.loc[peak_idx, "MINUTE_OF_DAY"] / 60.0)

    return out


# ==========================================
# PROBABLE REST WINDOW PROXY
# ==========================================
def compute_rest_proxy(day_df):
    out = {
        "REST_ONSET_H_PROXY": np.nan,
        "REST_OFFSET_H_PROXY": np.nan,
        "REST_MIDPOINT_H_PROXY": np.nan,
        "REST_DURATION_H_PROXY": np.nan,
        "TEMP_DURING_REST_PROXY": np.nan,
    }

    needed = [TEMP_COL, ACTIVITY_COL, LIGHT_COL]
    if not all(col in day_df.columns for col in needed):
        return out

    tmp = day_df[[DATETIME_COL, "MINUTE_OF_DAY", TEMP_COL, ACTIVITY_COL, LIGHT_COL]].dropna().copy()
    if len(tmp) == 0:
        return out

    act_thr = tmp[ACTIVITY_COL].quantile(REST_ACTIVITY_QUANTILE)
    light_thr = tmp[LIGHT_COL].quantile(REST_LIGHT_QUANTILE)
    temp_thr = tmp[TEMP_COL].quantile(REST_TEMP_QUANTILE)

    rest_mask = (
        (tmp[ACTIVITY_COL] <= act_thr) &
        (tmp[LIGHT_COL] <= light_thr) &
        (tmp[TEMP_COL] >= temp_thr)
    )

    rest = tmp.loc[rest_mask].copy()
    if len(rest) == 0:
        return out

    onset_h = float(rest["MINUTE_OF_DAY"].iloc[0] / 60.0)
    offset_h = float(rest["MINUTE_OF_DAY"].iloc[-1] / 60.0)

    # handle across-midnight durations
    duration = offset_h - onset_h
    if duration < 0:
        duration += 24
    midpoint = (onset_h + duration / 2) % 24

    out["REST_ONSET_H_PROXY"] = onset_h
    out["REST_OFFSET_H_PROXY"] = offset_h
    out["REST_MIDPOINT_H_PROXY"] = midpoint
    out["REST_DURATION_H_PROXY"] = duration
    out["TEMP_DURING_REST_PROXY"] = rest[TEMP_COL].mean()

    return out


# ==========================================
# DAILY CHRONOBIO SUMMARY
# ==========================================
def summarize_day_chronobio(day_df):
    out = {"N_OBS": int(day_df[TEMP_COL].notna().sum())}

    temp = day_df[TEMP_COL].dropna()
    if len(temp) == 0:
        return pd.Series(out)

    out["TEMP_MEAN"] = temp.mean()
    out["TEMP_SD"] = temp.std()
    out["TEMP_MIN"] = temp.min()
    out["TEMP_MAX"] = temp.max()
    out["TEMP_RANGE"] = temp.max() - temp.min()
    out["TEMP_P05"] = temp.quantile(0.05)
    out["TEMP_P95"] = temp.quantile(0.95)
    out["TEMP_ROBUST_AMPLITUDE"] = (out["TEMP_P95"] - out["TEMP_P05"]) / 2

    # Day-night contrast
    temp_day = day_df.loc[day_df["IS_DAY"] == 1, TEMP_COL].dropna()
    temp_night = day_df.loc[day_df["IS_NIGHT"] == 1, TEMP_COL].dropna()
    temp_evening = day_df.loc[day_df["IS_EVENING"] == 1, TEMP_COL].dropna()

    out["TEMP_DAY_MEAN"] = temp_day.mean() if len(temp_day) else np.nan
    out["TEMP_NIGHT_MEAN"] = temp_night.mean() if len(temp_night) else np.nan
    out["TEMP_EVENING_MEAN"] = temp_evening.mean() if len(temp_evening) else np.nan

    out["TEMP_NIGHT_MINUS_DAY"] = (
        out["TEMP_NIGHT_MEAN"] - out["TEMP_DAY_MEAN"]
        if pd.notna(out["TEMP_NIGHT_MEAN"]) and pd.notna(out["TEMP_DAY_MEAN"])
        else np.nan
    )

    # Evening rise relative to daytime baseline
    if len(temp_day) > 0 and len(temp_evening) > 0:
        out["TEMP_EVENING_RISE"] = temp_evening.mean() - temp_day.mean()
    else:
        out["TEMP_EVENING_RISE"] = np.nan

    # Warming onset
    if len(temp_day) > 0:
        warming_threshold = temp_day.mean() + 0.5 * temp.std() if pd.notna(temp.std()) else temp_day.mean()
        out["TEMP_WARMING_ONSET_H"] = get_first_time_above_threshold(
            day_df, TEMP_COL, warming_threshold, hour_start=18, hour_end=24
        )
    else:
        out["TEMP_WARMING_ONSET_H"] = np.nan

    # Extreme timing
    temp_max_val, temp_max_h = get_daily_extreme_time(day_df, TEMP_COL, mode="max")
    temp_min_val, temp_min_h = get_daily_extreme_time(day_df, TEMP_COL, mode="min")
    out["TEMP_MAX_H"] = temp_max_h
    out["TEMP_MIN_H"] = temp_min_h

    # Hottest and coldest windows
    m5, m5_h = get_window_mean_extreme_time(day_df, TEMP_COL, window_hours=5, mode="max")
    l10, l10_h = get_window_mean_extreme_time(day_df, TEMP_COL, window_hours=10, mode="min")
    out["TEMP_M5"] = m5
    out["TEMP_M5_H"] = m5_h
    out["TEMP_L10"] = l10
    out["TEMP_L10_H"] = l10_h
    out["TEMP_RA"] = (m5 - l10) / (m5 + l10) if pd.notna(m5) and pd.notna(l10) and (m5 + l10) != 0 else np.nan

    # Cosinor
    out.update(fit_24h_cosinor(day_df, TEMP_COL))

    # Activity/light coupling
    if ACTIVITY_COL in day_df.columns:
        out["TEMP_ACTIVITY_CORR"] = safe_corr(day_df[TEMP_COL], day_df[ACTIVITY_COL])

    if LIGHT_COL in day_df.columns:
        out["TEMP_LIGHT_CORR"] = safe_corr(day_df[TEMP_COL], day_df[LIGHT_COL])
        out["TEMP_LOG_LIGHT_CORR"] = safe_corr(day_df[TEMP_COL], np.log1p(day_df[LIGHT_COL]))

    # Light markers
    light_markers = compute_daily_light_markers(day_df)
    out.update(light_markers)

    # Activity markers
    activity_markers = compute_daily_activity_markers(day_df)
    out.update(activity_markers)

    # Rest proxy
    rest_markers = compute_rest_proxy(day_df)
    out.update(rest_markers)

    # Phase angles
    out["PHASE_TEMP_TO_LIGHT_PEAK_H"] = circular_diff_hours(out["TEMP_COS_ACROPHASE_H"], out["LIGHT_PEAK_H"])
    out["PHASE_TEMP_TO_ACTIVITY_PEAK_H"] = circular_diff_hours(out["TEMP_COS_ACROPHASE_H"], out["ACTIVITY_PEAK_H"])
    out["PHASE_WARMING_TO_LIGHT_OFFSET_H"] = circular_diff_hours(out["TEMP_WARMING_ONSET_H"], out["LIGHT_OFFSET_H"])
    out["PHASE_WARMING_TO_ACTIVITY_OFFSET_H"] = circular_diff_hours(out["TEMP_WARMING_ONSET_H"], out["ACTIVITY_OFFSET_H"])
    out["PHASE_TEMP_TO_REST_MIDPOINT_H"] = circular_diff_hours(out["TEMP_COS_ACROPHASE_H"], out["REST_MIDPOINT_H_PROXY"])

    # Alignment / misalignment proxies
    if pd.notna(out["TEMP_EVENING_RISE"]) and pd.notna(out["EVENING_LIGHT_SUPPRESSION_INDEX"]):
        out["LIGHT_ADJUSTED_TEMP_RISE"] = out["TEMP_EVENING_RISE"] / (1 + out["EVENING_LIGHT_SUPPRESSION_INDEX"])
    else:
        out["LIGHT_ADJUSTED_TEMP_RISE"] = np.nan

    return pd.Series(out)


def compute_daily_chronobio_summary(df):
    daily = (
        df.groupby("DATE", group_keys=False)
        .apply(summarize_day_chronobio)
        .reset_index()
    )

    if len(daily) == 0:
        return daily

    daily["DATE"] = pd.to_datetime(daily["DATE"])
    daily = daily.sort_values("DATE").reset_index(drop=True)

    # rolling baselines for deviation / anomaly
    for col in [
        "TEMP_MEAN",
        "TEMP_NIGHT_MINUS_DAY",
        "TEMP_EVENING_RISE",
        "TEMP_COS_ACROPHASE_H",
        "TEMP_COS_AMPLITUDE",
        "REST_MIDPOINT_H_PROXY",
    ]:
        if col in daily.columns:
            daily[f"{col}_BASELINE_7D"] = daily[col].rolling(7, min_periods=3).mean()
            daily[f"{col}_DELTA_7D"] = daily[col] - daily[f"{col}_BASELINE_7D"]

    # day-to-day phase shift
    if "TEMP_COS_ACROPHASE_H" in daily.columns:
        daily["TEMP_ACROPHASE_SHIFT_FROM_PREV_H"] = [
            np.nan
        ] + [
            circular_diff_hours(daily.loc[i, "TEMP_COS_ACROPHASE_H"], daily.loc[i - 1, "TEMP_COS_ACROPHASE_H"])
            for i in range(1, len(daily))
        ]

    if "REST_MIDPOINT_H_PROXY" in daily.columns:
        daily["REST_MIDPOINT_SHIFT_FROM_PREV_H"] = [
            np.nan
        ] + [
            circular_diff_hours(daily.loc[i, "REST_MIDPOINT_H_PROXY"], daily.loc[i - 1, "REST_MIDPOINT_H_PROXY"])
            for i in range(1, len(daily))
        ]

    daily["IS_WEEKEND"] = (daily["DATE"].dt.weekday >= 5).astype(int)

    return daily


# ==========================================
# OVERALL CHRONOBIO METRICS
# ==========================================
def compute_overall_temp_rhythm_metrics(df):
    s = (
        df.set_index(DATETIME_COL)[TEMP_COL]
        .resample("1h")
        .mean()
        .dropna()
    )

    out = {
        "N_HOURLY_POINTS": len(s),
        "TEMP_IS": np.nan,
        "TEMP_IV": np.nan,
        "TEMP_RA_GLOBAL": np.nan,
        "TEMP_M5_GLOBAL": np.nan,
        "TEMP_L10_GLOBAL": np.nan,
    }

    if len(s) < 48:
        return out

    x = s.to_numpy()
    N = len(x)
    grand_mean = np.mean(x)
    denom = np.sum((x - grand_mean) ** 2)

    # Interdaily Stability
    mean_by_hour = s.groupby(s.index.hour).mean()
    if denom > 0:
        out["TEMP_IS"] = (N * np.sum((mean_by_hour - grand_mean) ** 2)) / (24 * denom)

    # Intradaily Variability
    if denom > 0 and N > 1:
        out["TEMP_IV"] = (N * np.sum(np.diff(x) ** 2)) / ((N - 1) * denom)

    # Relative Amplitude
    m5 = s.rolling(5, min_periods=5).mean().max()
    l10 = s.rolling(10, min_periods=10).mean().min()

    out["TEMP_M5_GLOBAL"] = m5
    out["TEMP_L10_GLOBAL"] = l10
    if pd.notna(m5) and pd.notna(l10) and (m5 + l10) != 0:
        out["TEMP_RA_GLOBAL"] = (m5 - l10) / (m5 + l10)

    return out


def compute_weekday_weekend_comparison(daily):
    cols = [
        "TEMP_COS_ACROPHASE_H",
        "TEMP_COS_AMPLITUDE",
        "TEMP_NIGHT_MINUS_DAY",
        "TEMP_EVENING_RISE",
        "TEMP_WARMING_ONSET_H",
        "REST_MIDPOINT_H_PROXY",
        "LIGHT_OFFSET_H",
        "ACTIVITY_OFFSET_H",
        "PHASE_WARMING_TO_LIGHT_OFFSET_H",
    ]

    out = {}

    wd = daily[daily["IS_WEEKEND"] == 0]
    we = daily[daily["IS_WEEKEND"] == 1]

    for col in cols:
        if col not in daily.columns:
            continue

        out[f"{col}_WEEKDAY_MEAN"] = wd[col].mean() if len(wd) else np.nan
        out[f"{col}_WEEKEND_MEAN"] = we[col].mean() if len(we) else np.nan

        # For clock time variables use circular difference
        if col.endswith("_H") or "ACROPHASE" in col or "MIDPOINT" in col:
            out[f"{col}_WEEKEND_MINUS_WEEKDAY"] = circular_diff_hours(
                out[f"{col}_WEEKEND_MEAN"],
                out[f"{col}_WEEKDAY_MEAN"]
            )
        else:
            out[f"{col}_WEEKEND_MINUS_WEEKDAY"] = (
                out[f"{col}_WEEKEND_MEAN"] - out[f"{col}_WEEKDAY_MEAN"]
                if pd.notna(out[f"{col}_WEEKEND_MEAN"]) and pd.notna(out[f"{col}_WEEKDAY_MEAN"])
                else np.nan
            )

    return pd.DataFrame([out])


# ==========================================
# MAIN
# ==========================================
def main():
    outdir = Path(OUTPUT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)

    df = load()
    df = add_datetime_features(df)
    df = preprocess(df)
    df = add_pointwise_chronobio_features(df)

    daily = compute_daily_chronobio_summary(df)
    overall = pd.DataFrame([compute_overall_temp_rhythm_metrics(df)])
    weekday_weekend = compute_weekday_weekend_comparison(daily)

    # Save
    #df.to_csv(outdir / "data_with_chronobio_temp_features.csv", index=False)
    #daily.to_csv(outdir / "daily_temp_chronobio_summary.csv", index=False)
    #overall.to_csv(outdir / "overall_temp_rhythm_metrics.csv", index=False)
    #weekday_weekend.to_csv(outdir / "weekday_weekend_temp_comparison.csv", index=False)

    print("\nPointwise columns:")
    print(df.columns.tolist())

    print("\nDaily chronobiology summary head:")
    print(daily.head())

    print("\nOverall rhythm metrics:")
    print(overall)

    print("\nWeekday vs weekend comparison:")
    print(weekday_weekend)


if __name__ == "__main__":
    main()


Pointwise columns:
['DATE/TIME', 'MS', 'EVENT', 'TEMPERATURE', 'EXT.TEMPERATURE', 'ORIENTATION', 'PIM', 'PIMn', 'TAT', 'TATn', 'ZCM', 'ZCMn', 'LIGHT', 'AMB.LIGHT', 'RED.LIGHT', 'GREEN.LIGHT', 'BLUE.LIGHT', 'IR.LIGHT', 'UVA.LIGHT', 'UVB.LIGHT', 'STATE', 'CAP_SENS_1', 'CAP_SENS_2', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'MELANOPIC.EDI', 'CLEAR', 'SLEEP_STATE', 'DATE', 'TIME', 'WEEK', 'ISO_YEAR', 'DAY_OF_WEEK', 'WEEKDAY_NUM', 'IS_WEEKEND', 'HOUR', 'MINUTE', 'SECOND', 'MINUTE_OF_DAY', 'IS_DAY', 'IS_EVENING', 'IS_NIGHT', 'LOG_LIGHT', 'TEMP_SMOOTH_30MIN', 'TEMP_SMOOTH_60MIN', 'TEMP_SD_60MIN', 'ACT_SMOOTH_30MIN', 'ACT_SMOOTH_60MIN', 'LIGHT_SMOOTH_30MIN', 'LIGHT_SMOOTH_60MIN', 'LOG_LIGHT_SMOOTH_30MIN', 'TEMP_DAY_MEAN_POINTWISE', 'TEMP_DAY_SD_POINTWISE', 'TEMP_CENTERED_DAY', 'TEMP_Z_DAY', 'REST_PROPENSITY_PROXY']

Daily chronobiology summary head:
        DATE   N_OBS  TEMP_MEAN   TEMP_SD  TEMP_MIN  TEMP_MAX  TEMP_RANGE  \
0 2025-12-19  1440.0  29.720521  2.793689     18.38     35.56 

/var/folders/j5/c5vt47v9695609y1ssl7zjdr0000gn/T/ipykernel_39897/311028490.py:533: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_day_chronobio)
